<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

## Step 6.0 — Setup

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

# Folder + file constants. _FOLDER = a directory, _FILE = a single file.
TUTORIALS_FOLDER = Path("tutorials")
DATA_FOLDER = TUTORIALS_FOLDER / "data"
SNAPSHOT_FOLDER = DATA_FOLDER / "snapshots"
CROP_FOLDER = DATA_FOLDER / "crops"
DB_FILE = DATA_FOLDER / "birds.db"
SAMPLE_BIRD_FILE = DATA_FOLDER / "samples" / "sample-bird.jpg"
MODEL_FILE = TUTORIALS_FOLDER / "yolov8n.pt"

# From .env (or hardcoded fallback for Colab)
PHONE_IP = os.environ.get("PHONE_IP", "192.168.1.42")
PHONE_URL = f"http://{PHONE_IP}:8080/photo.jpg"
SLACK_WEBHOOK = os.environ.get("SLACK_WEBHOOK", "")
HUGGINGFACE_API_KEY = os.environ.get("HUGGINGFACE_API_KEY", "")

SNAPSHOT_FOLDER.mkdir(parents=True, exist_ok=True)
CROP_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Snapshot folder: {SNAPSHOT_FOLDER}")
print(f"Phone URL: {PHONE_URL}")

Snapshot folder: tutorials/data/snapshots
Phone URL: http://192.168.1.207:8080/photo.jpg


## Step 6.1 — A table for sightings

Each row is one sighting: when, what species, how sure we were, which photo file.

In [ ]:
import sqlite3

DB_FILE.parent.mkdir(parents=True, exist_ok=True)
with sqlite3.connect(DB_FILE) as conn:
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS sightings (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp       TEXT    NOT NULL,
            species         TEXT    NOT NULL,
            confidence      REAL    NOT NULL,
            snapshot_file   TEXT    NOT NULL,
            bbox_x_min      INTEGER,
            bbox_y_min      INTEGER,
            bbox_x_max      INTEGER,
            bbox_y_max      INTEGER
        )
        """
    )
    conn.commit()
print(f"Table ready at {DB_FILE}")

Table ready at tutorials/data/birds.db


## Step 6.2 — Insert a sighting (parameterized, never f-string)

Always use `?` placeholders for user-supplied data. F-strings in SQL = SQL injection vulnerabilities.

In [ ]:
from datetime import datetime

timestamp = datetime.now().isoformat(timespec="seconds")
with sqlite3.connect(DB_FILE) as conn:
    cursor = conn.execute(
        """
        INSERT INTO sightings (timestamp, species, confidence, snapshot_file)
        VALUES (?, ?, ?, ?)
        """,
        (timestamp, "Northern Cardinal", 0.92, "data/snapshots/2026-07-07_18-30-00.jpg"),
    )
    conn.commit()
    print("Inserted row id:", cursor.lastrowid)

Inserted row id: 1


## Step 6.3 — Wrap as `save_sighting_to_db`

In [0]:
#| echo: false
#| output: asis
show_doc(save_sighting_to_db)

---

### save_sighting_to_db

```python
def save_sighting_to_db(
    db_file:Path, snapshot_file:Path, species:str, confidence:float, bounding_box:dict | None=None,
    verbose:bool=True
)->int:
```

*Add one new sighting to the database.*

Args:
    db_file: path to the SQLite file.
    snapshot_file: path to the photo file this sighting came from.
    species: the bird's name.
    confidence: how sure the classifier was, 0.0 - 1.0.
    bounding_box: optional dict with x_min, y_min, x_max, y_max.
    verbose: if True, print the new row id. Default True.

Returns:
    The new row's id (auto-incremented).

## Step 6.4 — Add `list_sightings_from_db`

In [0]:
#| echo: false
#| output: asis
show_doc(list_sightings_from_db)

---

### list_sightings_from_db

```python
def list_sightings_from_db(
    db_file:Path, since:str | None=None, limit:int=100, verbose:bool=True
)->list:
```

*Get sightings from the database, most recent first.*

Args:
    db_file: path to the SQLite file.
    since: optional ISO timestamp; only return sightings after this.
    limit: max rows to return.
    verbose: if True, print a count. Default True.

Returns:
    List of sighting dicts, sorted by timestamp descending.

## Acceptance criterion

In [ ]:
from bird_watcher.save_sighting import save_sighting_to_db, list_sightings_from_db

jpg_files = sorted(SNAPSHOT_FOLDER.glob("*.jpg"))
snapshot_file = SNAPSHOT_FOLDER / jpg_files[0].name if jpg_files else SNAPSHOT_FOLDER / "missing.jpg"

row_id = save_sighting_to_db(DB_FILE, snapshot_file, "Northern Cardinal", 0.92)
rows = list_sightings_from_db(DB_FILE)
assert any(r["id"] == row_id for r in rows), "New row should be in the listing"
print(f"✅ {len(rows)} sighting(s), most recent id={rows[0]['id']}")

Saved sighting #2: Northern Cardinal (0.92)
2 sighting(s) in birds.db
✅ 2 sighting(s), most recent id=1


## What's next

**Step 7:** open [07-slack.ipynb](07-slack.ipynb) — we'll send a Slack message whenever we save a sighting.